In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score
from sklearn.manifold import TSNE
import time
import os
import train_utils


In [2]:
# 基础转换（计算统计特性用）
base_transform = transforms.Compose([transforms.ToTensor()])
train_data_raw = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=base_transform)

# 计算训练集通道均值和标准差
def get_dataset_stats(dataset):
    imgs = torch.stack([img for img, _ in dataset], dim=0)
    mean = imgs.view(3, -1).mean(dim=1).numpy()
    std = imgs.view(3, -1).std(dim=1).numpy()
    return mean, std

# 统计值为：mean=[0.4914, 0.4822, 0.4465], std=[0.2470, 0.2435, 0.2616]
# dataset_mean, dataset_std = get_dataset_stats(train_data_raw)
dataset_mean = [0.4914, 0.4822, 0.4465]
dataset_std = [0.2470, 0.2435, 0.2616]
print(f"Used Mean: {dataset_mean}, Std: {dataset_std}")

# 定义正式的 Transforms
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(dataset_mean, dataset_std)
])

val_test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(dataset_mean, dataset_std)
])

# 加载并划分数据集
full_train_set = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=train_transform)
train_size = int(0.9 * len(full_train_set))
val_size = len(full_train_set) - train_size
train_set, val_set = torch.utils.data.random_split(full_train_set, [train_size, val_size])

# 验证集应使用不带增强的 transform
# Note: Changing transform after split is tricky with random_split as it wraps the dataset
# Correct way is to create two datasets or use a wrapper. 
# But assuming the user wants to keep logic simple or fix it?
# In random_split, train_set and val_set are Subsets, which have .dataset attribute pointing to full_train_set.
# Changing full_train_set.transform changes it for BOTH. This is a common bug.
# However, user's code did: val_set.dataset.transform = val_test_transform
# This modifies full_train_set.transform! Which affects train_set too!
# I should fix this bug while I'm at it? Or just stick to refactoring?
# The user asked to refactor based on train_utils. Fixing a bug is good refactoring.

# Use separate datasets
train_set_full = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=train_transform)
val_set_full = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=val_test_transform) # Same data, different transform

indices = torch.randperm(len(train_set_full)).tolist()
train_indices = indices[:train_size]
val_indices = indices[train_size:]

train_set = Subset(train_set_full, train_indices)
val_set = Subset(val_set_full, val_indices)

test_set = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=val_test_transform)

# Use train_utils for dataloaders
train_loader, val_loader = train_utils.get_dataloaders(train_set, val_set, batch_size=128, num_workers=0)
# Create test_loader manually or reusing utils (but utils forces shuffle=True for first arg)
test_loader = DataLoader(test_set, batch_size=64, shuffle=False, num_workers=0)

Used Mean: [0.4914, 0.4822, 0.4465], Std: [0.247, 0.2435, 0.2616]


In [3]:
# ## 2. 手动构建 CNN (ResNet-20)
# 基于 He 等人 (2016) 的论文，针对 32x32 输入进行适配。采用 Option A 的捷径连接。

# %%
class BasicBlock(nn.Module):
    def __init__(self, in_planes, planes, stride=1):
        super(BasicBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)

        self.shortcut = nn.Sequential()
        # Option A: Stride=2 时进行下采样并用零填充增加维度
        if stride!= 1 or in_planes!= planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, planes, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(planes)
            )

    def forward(self, x):
        out = torch.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        out = torch.relu(out)
        return out

class ResNet20(nn.Module):
    def __init__(self, num_classes=10):
        super(ResNet20, self).__init__()
        self.in_planes = 16
        # 初始层
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(16)
        # 3个阶段，每个阶段3个块
        self.layer1 = self._make_layer(16, 3, stride=1)
        self.layer2 = self._make_layer(32, 3, stride=2)
        self.layer3 = self._make_layer(64, 3, stride=2)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(64, num_classes)

    def _make_layer(self, planes, num_blocks, stride):
        strides = [stride] + [1]*(num_blocks-1)
        layers = []
        for s in strides:
            layers.append(BasicBlock(self.in_planes, planes, s))
            self.in_planes = planes
        return nn.Sequential(*layers)

    def forward(self, x):
        out = torch.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.avgpool(out)
        feat = torch.flatten(out, 1) # Penultimate layer features
        logits = self.fc(feat)
        return logits, feat

In [4]:

# ## 3. 微调 Swin Transformer
# 加载 `swin_t` 预训练权重，调整输入分辨率为 224x224 并修改分类头。

# %%
from torchvision.models import swin_t, Swin_T_Weights

def get_swin_model(num_classes=10):
    weights = Swin_T_Weights.IMAGENET1K_V1
    model = swin_t(weights=weights)

    # 修改全连接层
    model.head = nn.Linear(model.head.in_features, num_classes)

    # 定义特定的预处理逻辑：Resize 32 -> 224
    swin_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    return model, swin_transform

In [5]:
# ## 4. 训练与评估逻辑
# 这里整合了 `train_utils.py` 要求的核心逻辑：训练循环、F1-Score 计算和损失图绘制。

# %%
# `train_utils.train_and_eval_visualized` 将用于替代 `train_model`
# def train_model(model, train_loader, val_loader, criterion, optimizer, epochs, device, model_name="model"):
#     ...

def final_test(model, test_loader, device):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            # Handle tuple output from ResNet20
            output = model(images)
            if isinstance(output, tuple):
                outputs, _ = output
            else:
                outputs = output
            
            _, predicted = outputs.max(1)
            y_true.extend(labels.cpu().numpy())
            y_pred.extend(predicted.cpu().numpy())

    acc = 100. * np.sum(np.array(y_true) == np.array(y_pred)) / len(y_true)
    f1 = f1_score(y_true, y_pred, average='macro') # Macro-average F1-Score
    return acc, f1

In [6]:

# ## 5. 特征可视化 (t-SNE)
# 采样 1,000 张测试集图像，提取倒数第二层特征进行降维。

# %%
def visualize_features(model, dataloader, device, n_samples=1000):
    model.eval()
    features, labels = [], []
    with torch.no_grad():
        for i, (imgs, lbls) in enumerate(dataloader):
            imgs = imgs.to(device)
            # 提取倒数第二层特征向量
            if isinstance(model, ResNet20):
                _, feat = model(imgs)
            else:
                # 对 Swin Transformer 提取 head 之前的特征
                feat = model.forward_features(imgs)
                feat = nn.AdaptiveAvgPool2d((1, 1))(feat).view(feat.size(0), -1)

            features.append(feat.cpu().numpy())
            labels.append(lbls.numpy())
            if len(np.concatenate(features)) >= n_samples: break

    X = np.concatenate(features)[:n_samples]
    y = np.concatenate(labels)[:n_samples]

    # t-SNE 降维
    tsne = TSNE(n_components=2, perplexity=30, random_state=42)
    X_embedded = tsne.fit_transform(X)

    plt.figure(figsize=(10, 8))
    scatter = plt.scatter(X_embedded[:, 0], X_embedded[:, 1], c=y, cmap='tab10', alpha=0.6)
    plt.colorbar(scatter)
    plt.title("t-SNE Visualization of Penultimate Layer Features")
    plt.show()

In [ ]:
# ## 6. 执行实验
# 设定超参数并启动流程 。

# %%
# 1. 设置设备
device = train_utils.get_device()

# 2. 训练 ResNet-20
print("\n=== Training ResNet-20 ===")
resnet = ResNet20().to(device)
optimizer_res = optim.SGD(resnet.parameters(), lr=0.1, momentum=0.9, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

# 使用工具库进行训练
train_utils.train_and_eval_visualized(
    model=resnet,
    train_loader=train_loader,
    test_loader=val_loader,
    optimizer=optimizer_res,
    criterion=criterion,
    epochs=60,
    save_dir='./checkpoints/resnet20',
    device=device,
    model_name="ResNet-20"
)

# 加载最佳模型进行最终测试
best_resnet_path = './checkpoints/resnet20/best_model.pth'
if os.path.exists(best_resnet_path):
    checkpoint = torch.load(best_resnet_path, map_location=device)
    resnet.load_state_dict(checkpoint['model_state_dict'])
    print("Loaded best ResNet-20 model.")

test_acc, test_f1 = final_test(resnet, test_loader, device)
print(f"ResNet-20 Test Accuracy: {test_acc:.2f}%, F1-Score: {test_f1:.4f}")
visualize_features(resnet, test_loader, device)

# 3. 训练 Swin-T
print("\n=== Fine-tuning Swin Transformer ===")
swin_model, swin_trans = get_swin_model()
swin_model = swin_model.to(device)

# 调整 DataLoader 以适应 Swin 的 224 输入
# 注意：这里重新创建 loader，但 dataset 需要应用 swin_trans
# 这是一个 tricky part：full_train_set 上次用了 train_transform (32x32)
# 需要一个新的 dataset 来做 Swin 的 transform
swin_train_set = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=swin_trans)
# 取前 5000 个
swin_train_subset = Subset(swin_train_set, range(5000))

# 同样需要对应的 val_loader
swin_val_set = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=swin_trans)
swin_val_subset = Subset(swin_val_set, val_indices[:1000]) # 示例取部分验证集以节省时间

swin_train_loader = DataLoader(swin_train_subset, batch_size=32, shuffle=True, num_workers=0)
swin_val_loader = DataLoader(swin_val_subset, batch_size=32, shuffle=False, num_workers=0)

optimizer_swin = optim.AdamW(swin_model.head.parameters(), lr=1e-3, weight_decay=0.05)

train_utils.train_and_eval_visualized(
    model=swin_model,
    train_loader=swin_train_loader,
    test_loader=swin_val_loader,
    optimizer=optimizer_swin,
    criterion=criterion,
    epochs=50, # 示例 epochs
    save_dir='./checkpoints/swin',
    device=device,
    model_name="Swin-Transformer"
)
test_acc, test_f1 = final_test(swin_model, swin_val_loader, device)
print(f"Swin-Transformer Test Accuracy: {test_acc:.2f}%, F1-Score: {test_f1:.4f}")
# visualize_features(swin_model, swin_val_loader, device)
